In [72]:
# Libraries
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [81]:
# Loading in csv from GitHub
df = pd.read_csv('https://raw.githubusercontent.com/jkenloh/visualization/refs/heads/assignment-3/02_activities/assignments/Dinesafe.csv')
df.head(3)

,_id,Establishment ID,Inspection ID,Establishment Name,Establishment Type,Establishment Address,Establishment Status,Min. Inspections Per Year,Infraction Details,Inspection Date,Severity,Action,Outcome,Amount Fined,Latitude,Longitude,unique_id
0,1,10657713,105133203.0,NEW KANTAMANTO MARKET,Food Depot,"266 EDDYSTONE AVE, Unit-0",Pass,2,FOOD PREMISE NOT MAINTAINED WITH CLEAN FLOORS ...,2023-03-07,M - Minor,Notice to Comply,NaN,NaN,43.74791,-79.52219,6e10cefe79756f0320205ba4eed824b0
1,2,10657713,105133203.0,NEW KANTAMANTO MARKET,Food Depot,"266 EDDYSTONE AVE, Unit-0",Pass,2,Operate food premise - equipment not arranged ...,2023-03-07,M - Minor,Notice to Comply,NaN,NaN,43.74791,-79.52219,ed41cbcc89db93061c463f66bf8a98cc
2,3,10657713,105238109.0,NEW KANTAMANTO MARKET,Food Depot,"266 EDDYSTONE AVE, Unit-0",Pass,2,NaN,2023-08-25,NaN,NaN,NaN,NaN,43.74791,-79.52219,5d3e7e93d5968017337246a9ee547631


In [ ]:
# Convert column "Inspection Date" to datetime
df["Inspection Date"] = pd.to_datetime(df["Inspection Date"])

# Extract Year
df["Year"] = df["Inspection Date"].dt.year

# Filter for Severity - Significant and Crucial infractions
df_filtered = df[df["Severity"].isin(["S - Significant", "C - Crucial"])]

# Group by
infraction_counts = (
    df_filtered.dropna(subset=["Infraction Details"])
    .groupby(["Year", "Establishment Name"])
    .size()
    .reset_index(name="Infraction Count")
)

# Years
years = sorted(infraction_counts["Year"].dropna().unique())

# Max infraction across all years - for x-axis
max_x_values = {
    year: infraction_counts[infraction_counts["Year"] == year]["Infraction Count"].max()
    for year in years
}
global_max_x = max(max_x_values.values())  

fig = go.Figure()

# Frames for each year
frames = []
for year in years:
    yearly_data = (
        infraction_counts[infraction_counts["Year"] == year]
        .sort_values(by="Infraction Count", ascending=False)
        .head(20)
    )

    # Highlight - top 5
    colors = ["#E69F00" if i < 5 else "#0072B2" for i in range(len(yearly_data))]

    frames.append(
        go.Frame(
            data=[
                go.Bar(
                    x=yearly_data["Infraction Count"],
                    y=yearly_data["Establishment Name"],
                    orientation="h",
                    marker=dict(color=colors),
                )
            ],
            name=str(year),
            layout=dict(xaxis=dict(range=[0, global_max_x + 2]))
        )
    )

# Initial Data
first_year_data = (
    infraction_counts[infraction_counts["Year"] == years[0]]
    .sort_values(by="Infraction Count", ascending=False)
    .head(20)
)

# Initial chart
fig.add_trace(
    go.Bar(
        x=first_year_data["Infraction Count"],
        y=first_year_data["Establishment Name"],
        orientation="h",
        marker=dict(color=colors),  
    )
)

fig.frames = frames

# Update layout
fig.update_layout(
    title="Top 20 Establishments with Significant/Crucial Infractions per Year",
    xaxis_title="Number of Infractions",
    yaxis_title="",
    xaxis=dict(range=[0, global_max_x + 2], fixedrange=True), 
    # Sort by most infractions at the top
    yaxis=dict(autorange="reversed"),
    height=600,
    width=1000,
    # Set specific margin to prevent x-axis from shifting due to y-axis name lengths
    margin=dict(l=350, r=50, t=50, b=50),
    uniformtext=dict(minsize=10, mode="hide"),
    bargap=0.1,
    # Remove any floating menus/buttons
    updatemenus=[],
    # Slider by year
    sliders=[
        {
            "active": 0,
            "yanchor": "top",
            "xanchor": "left",
            "currentvalue": {"font": {"size": 20}, "prefix": "Year: ", "visible": True, "xanchor": "right", "suffix": "",},
            "transition": {"duration": 0, "easing": "linear"},
            "pad": {"b": 10, "t": 50},
            "len": 0.9,
            "x": 0.1,
            "y": 0,
            "steps": [
                {
                    "args": [[str(year)], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
                    "label": str(int(year)),
                    "method": "animate",
                }
                for year in years
            ],
        }
    ],
)

fig.show()